### RAG Pipelines - Data ingestion to Vector DB pipeline

In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
from pathlib import Path

In [12]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: aws_learning_summary.pdf
  ✓ Loaded 13 pages

Processing: CloudNetworking_Session.pdf
  ✓ Loaded 9 pages

Processing: networking_deep_dive.pdf
  ✓ Loaded 12 pages

Total documents loaded: 34


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'aws_learning_summary.pdf', 'file_type': 'pdf'}, page_content='AWS Fundamentals and Cloud \nPractitioner \nLearning Summary \nInternal Training Documentation \n \nName \nPiyush Kumar \nEmployee ID \n513558 \nTeam / Department \nCloud and Data Center \nDocument Date \n6 August 2026'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 1, 'page_label': '2', 'source_file': 'aws_learning_summary.pdf', 'file

In [14]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
chunks = split_documents(all_pdf_documents)

Split 34 documents into 104 chunks

Example chunk:
Content: AWS Fundamentals and Cloud 
Practitioner 
Learning Summary 
Internal Training Documentation 
 
Name 
Piyush Kumar 
Employee ID 
513558 
Team / Department 
Cloud and Data Center 
Document Date 
6 Augus...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'aws_learning_summary.pdf', 'file_type': 'pdf'}


In [16]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'aws_learning_summary.pdf', 'file_type': 'pdf'}, page_content='AWS Fundamentals and Cloud \nPractitioner \nLearning Summary \nInternal Training Documentation \n \nName \nPiyush Kumar \nEmployee ID \n513558 \nTeam / Department \nCloud and Data Center \nDocument Date \n6 August 2026'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 1, 'page_label': '2', 'source_file': 'aws_learning_summary.pdf', 'file

### Embedding and VectorStoreDB

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


a:\langchainupdated\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\piyushp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2774.37it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\piyushp\AppData\Local\Temp\ipykernel_30716\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [4]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [17]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'aws_learning_summary.pdf', 'file_type': 'pdf'}, page_content='AWS Fundamentals and Cloud \nPractitioner \nLearning Summary \nInternal Training Documentation \n \nName \nPiyush Kumar \nEmployee ID \n513558 \nTeam / Department \nCloud and Data Center \nDocument Date \n6 August 2026'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-12T11:48:17+05:30', 'author': 'Un-named', 'moddate': '2026-08-12T11:48:17+05:30', 'source': '..\\data\\pdf\\aws_learning_summary.pdf', 'total_pages': 13, 'page': 1, 'page_label': '2', 'source_file': 'aws_learning_summary.pdf', 'file

In [18]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 104 texts...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.39it/s]


Generated embeddings with shape: (104, 384)
Adding 104 documents to vector store...
Successfully added 104 documents to vector store
Total documents in collection: 104


### Retriever Pipeline from VectorStore

In [19]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [20]:
rag_retriever

In [21]:
rag_retriever.retrieve("What is Corporate Network Technologies and Solutions.")

Retrieving documents for query: 'What is Corporate Network Technologies and Solutions.'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_75453170_39',
  'content': '5. Corporate Network Technologies and Solutions ................................ ................................ ............ 6 \n6. Core Network Concepts ................................ ................................ ................................ ............. 7 \n7. Key Takeaways ................................ ................................ ................................ ........................ 8 \n8. Conclusion ................................ ................................ ................................ ...............................  8',
  'metadata': {'creator': 'Microsoft® Word for Microsoft 365',
   'page': 1,
   'source_file': 'CloudNetworking_Session.pdf',
   'producer': 'Microsoft® Word for Microsoft 365',
   'doc_index': 39,
   'moddate': '2026-08-05T13:44:14+05:30',
   'total_pages': 9,
   'content_length': 562,
   'file_type': 'pdf',
   'creationdate': '2026-08-05T13:44:14+05:30',
   'source': '..\\data\\pdf\\Clo

In [22]:
rag_retriever.retrieve("Segmentation and Policy-Based Routing")

Retrieving documents for query: 'Segmentation and Policy-Based Routing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_8b6f6cdb_93',
  'content': 'region) \nProvides dedicated, private, high-bandwidth connectivity \nfrom on-premises data centres. \nTGW Route Table \nAttachments \nMultiple Legacy Transit Gateway route tables peered into the Cloud \nWAN for continued connectivity during and after migration. \nIPsec Site-to-Site \nVPN \nMultiple Encrypted tunnels for connecting external or partner \nnetworks to the Cloud WAN. \n \n5.4 Segmentation and Policy-Based Routing \nOne of the most powerful features of Cloud WAN is its ability to enforce network segmentation \nthrough segments and policies. In the Autodesk environment, VPCs are grouped into segments \nbased on their environment type (e.g., dev, stage, prod). [2] \n• Segments — Logical groupings of VPCs and attachments. VPCs within the same segment \ncan communicate with each other. Cross-segment communication is blocked by default. \n• Automatic Segment Assignment — VPCs are not manually assigned to segments.',
  'metadata': {'produce

### Integration VectorDB Context pipeline with LLM Output

In [35]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [36]:
answer=rag_simple("What is Cloud WAN?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Cloud WAN?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.91it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Cloud WAN** is a managed AWS service that lets organizations centrally build, manage, and monitor a global wide‑area network. It automatically provisions AWS resources (e.g., Transit Gateways, core network edges) across multiple regions, integrates with AWS Network Manager for a unified view, and provides policy‑based control for multi‑region and on‑premises workloads.


### Enhanced RAG Pipeline Features

In [37]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Palo Alto Networks ?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Palo Alto Networks ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: Palo Alto Networks is a leading enterprise cybersecurity company that specializes in network security and firewall technology, offering next‑generation firewalls, cloud‑delivered security services, and threat intelligence solutions.
Sources: [{'source': 'CloudNetworking_Session.pdf', 'page': 6, 'score': 0.2664531469345093, 'preview': "5.2 Palo Alto Networks \nPalo Alto Networks is a leading provider of enterprise cybersecurity solutions, with a strong focus on \nnetwork security and firewall technology. During the sessions, Palo Alto was referenced in the context of \nAutodesk's corporate network security infrastructure. At a genera..."}]
Confidence: 0.2664531469345093
Context Preview: 5.2 Palo Alto Networks 
Palo Alto Networks is a leading provider of enterprise cybersecurity solutions, with a strong focus on 
network security and firewall technology. During the sessions, Palo Alto was referenced in the context of 
Autodesk's corporate network security infrastructure. At a gen

### Advanced RAG Pipeline

In [38]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is AWS Cloud WAN?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


Retrieving documents for query: 'what is AWS Cloud WAN?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
5.3 AWS Cloud WAN 
AWS Cloud WAN (Wide Area Network) is a managed service that enables organizations to centrally build, 
manage, and monitor a global wide-area network us

ing AWS's infrastructure. It allows network engineers 
to define network policies through a central dashbo ard and automatically provisions the necessary AWS 
resources — such as Transit Gateways and core network edges — across multiple regions. 
Cloud WAN simplifies the process of building and maintaining complex multi-region network architectures 
by abstracting the underlying infrastructure. It integrates natively with AWS Network Manager, providing 
a consolidated view of the global network along side policy management capabilities. For organizations 
with workloads spanning multiple AWS regions and on-premises sites, Cloud WAN offers a scalable and 
operationally efficient connectivity solution. 
 
6. Core Network Concepts

with workloads spanning multiple AWS regions and on-premises sites, Cloud WAN offers a scalable and 
operationally efficient connectivity solution. 
 
6. Core Network Concepts 
 
 
The sessions also covered three foundational concepts relevant to the AWS Cloud 